# Experimental assays vs. computational predictors — four gene panels


In [ ]:
import sys
import yaml
import numpy as np
import polars as pl
from scipy import stats

from plotnine import *
import matplotlib.pyplot as plt
plt.rcParams['svg.fonttype'] = 'none'

## Parameters

In [ ]:
from pathlib import Path
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'utils' / 'variant_filtering.py').exists())

MASTER_PATH  = '../../../data/ukbbgym_genebass_all_20260824.parquet'
PROTEINGYM   = '../../../data/proteingym_SNVs_with_readout_annotated_20260716.parquet'
MARSH        = '../../../data/DMS_Marsh_VEP.parquet'
ROTH_LDLR    = '../../../data/LDLR_Roth_Science_2025.parquet'
CONFIG_DIR   = str(REPO_ROOT / 'configs')
FIG_DIR      = '../../../paper_figures'

CONFIG_FILE         = 'config_correlations.yaml'
SELECTED_CATEGORIES = ['missense', 'conservation', 'genetic_diversity', 'gnomad']
EXCLUDE_ANNOS       = []          # e.g. ['gnomade_af', 'gnomadg_af'] to drop the AF baselines
MAC                 = 20          # MAC <= 20 proxy on the Genebass betas
ONLY_SNPS           = True

N_BOOT     = 10000    # bootstrap replicates for the pairwise significance test
SEED       = 0
STAR_ALPHA = 0.05    # one-sided p < STAR_ALPHA per pair, raw (no multiple-testing correction --
                      # see the note in the test cell below for why an item that must beat every
                      # opposite-category item doesn't need one)

PTEN_INCLUDE_ABUNDANCE = False    # see the note above -- costs 53 of 101 variants

In [ ]:
# One entry per facet. `source` picks the reader below; `column` is the assay's name inside that
# file; `direction` is -1 when a high assay score means *more* function (i.e. less damaging).
#
# `phenotype` selects the trait when the master table holds several for a gene. Leave it None to
# take the table's own pick (largest |loftee_corr|, which is all the 2026-07-25 table stores).
PANELS = {
    'LDLR': dict(
        region='ENSG00000130164',
        # The 2026-07-25 table only has total_bilirubin_int for LDLR; with the full table set this
        # to 'ldl_direct_int' (or 'apolipoprotein_b_int') -- see the note at the top.
        phenotype='ldl_direct_int',
        assays=[
            dict(source='roth', column='LDL_uptake_functional',
                 label='LDL uptake (Roth 2025)'),
            dict(source='roth', column='LDLR_cell_surface_abundance',
                 label='LDLR surface abundance (Roth 2025)'),
        ],
    ),
    'GCK': dict(
        region='ENSG00000106633',
        phenotype='glycated_haemoglobin_hba1c_int',
        assays=[
            dict(source='proteingym', column='HXK4_HUMAN_Gersing_2022_activity',
                 label='GCK activity (Gersing 2022)'),
            dict(source='proteingym', column='HXK4_HUMAN_Gersing_2023_abundance',
                 label='GCK abundance (Gersing 2023)'),
        ],
    ),
    'PTEN': dict(
        region='ENSG00000171862',
        phenotype='igf1_int',
        assays=[
            dict(source='marsh', column='DMS_b',
                 label='PTEN activity (Mighell 2018)'),
        ],
    ),
    'PPARG': dict(
        region='ENSG00000132170',
        phenotype='hdl_cholesterol_int',
        assays=[
            dict(source='proteingym', column='PPARG_HUMAN_Majithia_2016',
                 label='PPARG activity (Majithia 2016)'),
        ],
        # MSA Pairformer barely scored this gene (11 of 36,815 PPARG variants have a score in the
        # source table) -- the few that survive into this panel don't give `n_unique() > 1` a
        # clean constant-column read below, so the column slips past the auto-drop with NaNs in
        # it and poisons rankdata/corrcoef for the whole panel. Drop it here explicitly.
        exclude_models=['msa_pairformer_llr'],
    ),
}

# Trait names for the facet strips (master table phenotype -> display label)
TRAIT_LABELS = {
    'total_bilirubin_int': 'total bilirubin',
    'glycated_haemoglobin_hba1c_int': 'HbA1c',
    'igf1_int': 'IGF-1',
    'hdl_cholesterol_int': 'HDL',
    'ldl_direct_int': 'LDL',
    'apolipoprotein_b_int': 'ApoB',
}

## Loading

`load_config` / `scan_variants` / `appv_of` are the master-table helpers from
[`master_table_analyses.ipynb`](../master_table_analyses.ipynb). The assay readers are new: each
returns a frame keyed the way its source can actually be keyed — ProteinGym on genomic
coordinates, the Marsh and Roth files on `(region, ref_aa, aa_position, alt_aa)`, since those
carry no coordinates.

In [ ]:
def load_config(config_file):
    """config_*.yaml -> (anno_config_df, all_annotation_list)."""
    cfg = yaml.safe_load(open(f'{CONFIG_DIR}/{config_file}'))
    df = pl.DataFrame([
        {'category': c, 'annotation': a, 'color': p['color'], 'label': p['label'],
         'annotation_dir': p.get('direction', 1)}
        for c, annos in cfg['rare_variant_annotations'].items() for a, p in annos.items()
    ]).with_columns(pl.col('annotation_dir').cast(pl.Int8))
    return df, df['annotation'].to_list()


COORD_KEY = ['region', 'chrom', 'pos', 'ref', 'alt']
AA_KEY    = ['region', 'ref_aa', 'aa_position', 'alt_aa']


def scan_master(regions, models):
    """Missense variants of `regions` that carry a Genebass beta at the MAC proxy.

    Both join keys are materialised here: the genomic coordinate and the amino-acid change
    parsed out of VEP's `amino_acids` / `protein_position`.
    """
    f = [pl.col('region').is_in(regions),
         pl.col('consequence_missense_variant') == True,
         pl.col('mean_pheno_value').is_not_null(),
         pl.col('AF') <= MAC / (2 * pl.col('n_cases')),
         pl.col('amino_acids').str.contains('/'),
         ~pl.col('protein_position').str.contains('-')]
    if ONLY_SNPS:
        f.append((pl.col('ref').str.len_chars() == 1) & (pl.col('alt').str.len_chars() == 1))
    return (pl.scan_parquet(MASTER_PATH).filter(*f)
        .with_columns(ref_aa=pl.col('amino_acids').str.split('/').list.get(0),
                      alt_aa=pl.col('amino_acids').str.split('/').list.get(1),
                      aa_position=pl.col('protein_position').str.split('/').list.get(0).cast(pl.Int64))
        .select(list(dict.fromkeys(COORD_KEY + AA_KEY +
                ['id', 'phenotype', 'mean_pheno_value', 'SE', 'loftee_corr', 'loftee_corr_dir']
                + models)))
        .unique()
        .collect(engine='streaming'))


def read_assay(spec, region):
    """One assay -> (key columns, frame with the assay score renamed to its `column`)."""
    src, col = spec['source'], spec['column']
    if src == 'proteingym':
        df = (pl.scan_parquet(PROTEINGYM)
              .filter(pl.col('region') == region, pl.col('file_name') == col)
              .select(COORD_KEY + [pl.col('dms_score').alias(col)]))
        key = COORD_KEY
    elif src == 'marsh':
        df = (pl.scan_parquet(MARSH)
              .filter(pl.col('region') == region, pl.col('DMS_assay') == col)
              .select(AA_KEY + [pl.col('DMS_score').alias(col)]))
        key = AA_KEY
    elif src == 'roth':
        df = (pl.scan_parquet(ROTH_LDLR)
              .filter(pl.col('region') == region, pl.col('assay') == col)
              .select(AA_KEY + [pl.col('score').alias(col)]))
        key = AA_KEY
    else:
        raise ValueError(f'unknown assay source: {src}')
    return key, df.drop_nulls().unique(subset=key).collect(engine='streaming')


def pick_phenotype(df, gene, spec):
    """One trait per panel. `phenotype=None` falls back to the largest |loftee_corr|."""
    want, avail = spec.get('phenotype'), df['phenotype'].unique().to_list()
    if want is None:
        if len(avail) > 1:
            want = (df.select(['phenotype', 'loftee_corr']).unique()
                      .sort(pl.col('loftee_corr').abs(), descending=True)['phenotype'][0])
            print(f'{gene}: {len(avail)} traits in the master table, taking |loftee_corr| winner '
                  f'{want!r} -- set PANELS[{gene!r}]["phenotype"] to override')
        else:
            want = avail[0]
    elif want not in avail:
        raise ValueError(f'{gene}: {want!r} not in the master table for this gene; available: {sorted(avail)}')
    return df.filter(pl.col('phenotype') == want)


def noise_ceiling(beta, se):
    """r_max: the correlation a perfect predictor would reach given the betas' own noise.

    Var(beta_hat) = Var(X) + mean(SE^2), so r_max = sqrt(Var(X) / (Var(X) + mean(SE^2))).
    Var(X) is Paule-Mandel, reused from the noise-ceiling analysis. Read it with the caveat
    from `noise_ceiling/02_ceiling.ipynb`: truncation at zero makes r_max ~0.1 even under no
    signal at these k, so only clearly larger values mean anything.
    """
    ok = np.isfinite(beta) & np.isfinite(se) & (se > 0)
    beta, se2 = beta[ok], se[ok] ** 2
    var_x = cu.paule_mandel(beta, se2)
    return float(np.sqrt(var_x / (var_x + se2.mean())))

In [ ]:
anno_config_df, all_annotation_list = load_config(CONFIG_FILE)
master_cols = pl.scan_parquet(MASTER_PATH).collect_schema().names()

model_cfg = (anno_config_df
             .filter(pl.col('category').is_in(SELECTED_CATEGORIES),
                     ~pl.col('annotation').is_in(EXCLUDE_ANNOS),
                     pl.col('annotation').is_in(master_cols))
             .unique(subset='annotation', maintain_order=True))
MODELS = model_cfg['annotation'].to_list()

missing = [a for a in anno_config_df.filter(pl.col('category').is_in(SELECTED_CATEGORIES))['annotation']
           if a not in master_cols]
print(f'{len(MODELS)} computational predictors; not in the master table: {missing}')

master = scan_master([p['region'] for p in PANELS.values()], MODELS)
print(f'master table: {master.height} missense variants with a beta across the {len(PANELS)} genes')
master.group_by(['region', 'phenotype']).len().sort('region')

### Per-panel variant sets

A panel's variant set is the master-table variants that **all** of that panel's assays cover, so
every bar in a facet — assay or predictor — is computed on exactly the same variants. The
coverage table below is also the isoform-mismatch alarm: an amino-acid join against the wrong
transcript collapses `matched` to near zero.

In [ ]:
panels, coverage = {}, []
for gene, spec in PANELS.items():
    df = pick_phenotype(master.filter(pl.col('region') == spec['region']), gene, spec)
    n_master = df.height
    for a in spec['assays']:
        key, assay_df = read_assay(a, spec['region'])
        df = df.join(assay_df, on=key, how='left')
        coverage.append({'gene': gene, 'assay': a['column'], 'key': 'coords' if key is COORD_KEY else 'aa',
                         'in_file': assay_df.height, 'master_variants': n_master,
                         'matched': n_master - df[a['column']].null_count()})
    cols = [a['column'] for a in spec['assays']]
    panels[gene] = df.drop_nulls(cols)
    d = panels[gene]
    print(f"{gene:6s} {spec['region']}  {d['phenotype'][0]:32s} n = {d.height:4d}  "
          f"(assays: {', '.join(cols)})")

pl.DataFrame(coverage)

## Correlations and the all-vs-all significance test

In [ ]:
def damaging_ranks(values, direction):
    """1..n ranks with the damaging end high, ties averaged (direction-corrected).

    Used only by the per-variant diagnostic scatter further down -- the significance test itself
    works on Spearman rho directly (`bootstrap_pairwise_pvals`), not on rank error.
    """
    return stats.rankdata(np.asarray(values, dtype=float) * direction)


def bootstrap_pairwise_pvals(X, y, n_boot=N_BOOT, seed=SEED):
    """One-sided paired-bootstrap p-value for every ordered pair of items in a panel.

    p[i, j] = P(item i fails to beat item j on Spearman rho with the trait), estimated by
    resampling variants with replacement and recomputing every item's rho against the same
    resampled y each time -- item i and item j are scored on the identical resample, so the
    variant-level noise they share cancels in the i-vs-j delta, the same way it would in a paired
    test. Makes no distributional assumption; the cost is a resolution floor of 1/(n_boot+1).

    H1 for cell (i, j): rho_i > rho_j. Diagonal is NaN (an item is never compared to itself).
    """
    rng = np.random.default_rng(seed)
    n, k = X.shape
    fails = np.zeros((k, k))     # fails[i, j]: replicates where i did NOT beat j
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        Rx = stats.rankdata(X[idx], axis=0)
        ry = stats.rankdata(y[idx])
        Rx = Rx - Rx.mean(axis=0)
        ry = ry - ry.mean()
        denom = np.sqrt((Rx ** 2).sum(axis=0) * (ry ** 2).sum())
        r = np.divide(Rx.T @ ry, denom, out=np.zeros(k), where=denom > 0)
        fails += (r[:, None] <= r[None, :])
    p = (fails + 1) / (n_boot + 1)
    np.fill_diagonal(p, np.nan)
    return p

In [ ]:
labels = {a: l.replace('\n', ' ') for a, l in model_cfg.select('annotation', 'label').iter_rows()}
dirs   = dict(model_cfg.select('annotation', 'annotation_dir').iter_rows())

corr_rows, pair_rows, rank_errors, orientation_failures = [], [], {}, []
for gene, spec in PANELS.items():
    d = panels[gene]
    trait = d['phenotype'][0]
    assays = [a['column'] for a in spec['assays']]
    for a in spec['assays']:
        labels[a['column']] = a['label']
        dirs[a['column']] = a.get('direction', -1)

    gene_models = [c for c in MODELS if c not in spec.get('exclude_models', [])]
    excluded = [c for c in MODELS if c in spec.get('exclude_models', [])]
    if excluded:
        print(f'{gene}: excluded ({", ".join(excluded)})')

    # A predictor with no spread over this panel's variants has no correlation to report.
    # In the master table that means the tool never scored this gene: PPARG's AlphaMissense
    # and CPT-1 columns are all-zero (and not even flagged by their `_is_na` column).
    items = [c for c in assays + gene_models if d[c].n_unique() > 1]
    dropped = [c for c in assays + gene_models if c not in items]
    if dropped:
        print(f'{gene}: dropped, constant over the panel: {dropped}')
    assays = [c for c in assays if c in items]
    tool_type = ['Experimental' if c in assays else 'Computational' for c in items]
    n = d.height

    # Sign is handled here and only here: scores are oriented so higher = more damaging
    # (`annotation_dir`), the beta so higher = the direction LoF variants move the trait
    # (`loftee_corr_dir`). Every rho below is therefore *signed* -- no abs() anywhere.
    y = stats.rankdata(d['mean_pheno_value'].to_numpy()) * d['loftee_corr_dir'][0]
    X = np.column_stack([stats.rankdata(d[c].to_numpy()) * dirs[c] for c in items])
    R = np.corrcoef(np.column_stack([y, X]), rowvar=False)   # row/col 0 is the trait
    r_trait = R[0, 1:]                                       # every item's rho with the trait, signed

    # --- ORIENTATION ASSERTION: full data only, never inside a bootstrap replicate ----------
    # Orientation is already applied upstream, so a negative rho here does not mean "a weak
    # predictor" -- it means the convention broke for this item: either its `direction` in
    # config_expAssays.yaml has the wrong sign, or `loftee_corr_dir` is inverted for this trait.
    # Both are upstream bugs worth finding, so the item is FAILED, never flipped or patched here.
    # (A negative rho inside a bootstrap replicate is expected and left signed.)
    orientation_ok = r_trait >= 0
    for i, item in enumerate(items):
        if not orientation_ok[i]:
            orientation_failures.append({'gene': gene, 'phenotype': trait, 'item': item,
                                         'label': labels[item], 'tool_type': tool_type[i],
                                         'rho': float(r_trait[i])})
            print(f'!! ORIENTATION FAILURE  {gene} / {trait}  {tool_type[i]} {labels[item]!r} '
                  f'({item}): rho = {r_trait[i]:+.4f} < 0 after upstream sign correction. Item '
                  f'failed, not flipped -- check its `direction` in {CONFIG_FILE} and '
                  f'`loftee_corr_dir` for this trait.')

    # Per-variant rank errors, for the pairwise diagnostic scatter further down only -- the
    # significance test below works on rho directly, not on rank error.
    true_rank = damaging_ranks(d['mean_pheno_value'].to_numpy(), d['loftee_corr_dir'][0])
    rank_errors[gene] = pl.DataFrame(
        {'id': d['id'], 'true_rank': true_rank}
        | {c: np.abs(damaging_ranks(d[c].to_numpy(), dirs[c]) - true_rank) for c in items})

    # All methods against all methods: one-sided bootstrap p-value for every ordered pair.
    P = bootstrap_pairwise_pvals(X, y)

    for i, item in enumerate(items):
        opp = [j for j in range(len(items)) if tool_type[j] != tool_type[i]]
        # A star needs item i to beat EVERY item of the opposite category -- an
        # intersection-union test: the joint claim "beats all opponents" is accepted only if
        # every individual one-sided test is significant. An IUT controls its overall false-
        # positive rate at STAR_ALPHA without any multiple-testing correction, because the joint
        # claim can only be wrongly accepted if EVERY individual (uncorrected) test is wrongly
        # significant -- which is already at most as likely as its least-significant component.
        worst_pval = float(np.max(P[i, opp])) if opp else np.nan
        beats_all  = bool(opp) and all(r_trait[i] > r_trait[j] for j in opp)
        oriented   = bool(orientation_ok[i]) and all(bool(orientation_ok[j]) for j in opp)
        star = bool(oriented and beats_all and worst_pval < STAR_ALPHA)

        corr_rows.append({'gene': gene, 'phenotype': trait, 'n_variants': n,
                          'item': item, 'label': labels[item], 'corr': r_trait[i],
                          'tool_type': tool_type[i], 'orientation_ok': bool(orientation_ok[i]),
                          'n_opponents': len(opp), 'worst_pval_vs_opposite': worst_pval,
                          'star': star})

        for j in range(len(items)):
            if j == i:
                continue
            pair_rows.append({'gene': gene, 'phenotype': trait,
                              'item_a': item, 'label_a': labels[item], 'type_a': tool_type[i],
                              'item_b': items[j], 'label_b': labels[items[j]], 'type_b': tool_type[j],
                              'rho_a': r_trait[i], 'rho_b': r_trait[j], 'delta': r_trait[i] - r_trait[j],
                              'pval': P[i, j], 'n_variants': n,
                              'orientation_ok': bool(orientation_ok[i] and orientation_ok[j]),
                              'a_beats_b': bool(r_trait[i] > r_trait[j] and P[i, j] < STAR_ALPHA
                                                 and orientation_ok[i] and orientation_ok[j])})

corr_df = pl.DataFrame(corr_rows)
pair_df = pl.DataFrame(pair_rows)

if orientation_failures:
    fail_df = pl.DataFrame(orientation_failures)
    print('\n' + '=' * 98)
    print(f'ORIENTATION ASSERTION FAILED for {fail_df.height} of {corr_df.height} items, across '
          f'{fail_df["gene"].n_unique()} panels. Excluded from starring; signs left untouched.')
    print('=' * 98)
    with pl.Config(tbl_rows=50, fmt_str_lengths=40):
        display(fail_df.sort('rho'))
else:
    print('orientation assertion passed: every rho >= 0 on the full data')

n_failed = int((~corr_df['orientation_ok']).sum())
n_star = int(corr_df['star'].sum())
print(f"\n{n_star} of {corr_df.height - n_failed} usable items beat every item of the opposite "
      f"category (bootstrap, one-sided, p < {STAR_ALPHA}, no multiple-testing correction -- see "
      "the note above)"
      + (f'; {n_failed} items failed the orientation assertion' if n_failed else ''))
corr_df.filter(pl.col('star')).sort(['gene', 'worst_pval_vs_opposite'])

In [ ]:
# All methods against all methods: one row per ordered pair within a gene panel.
pair_df.sort(['gene', 'type_a', 'rho_a'], descending=[False, False, True]).select(
    ['gene', 'label_a', 'type_a', 'label_b', 'type_b', 'rho_a', 'rho_b', 'delta', 'pval',
     'orientation_ok', 'a_beats_b'])

## Figure

One facet per gene, bars ordered by correlation within the facet (best on the right). The x-axis
key carries a `gene||` prefix so each facet can have its own ordering — the prefix is stripped
again by the axis labeller.

In [ ]:
plot_df = (corr_df
    .with_columns(plot_key=pl.col('gene') + '||' + pl.col('label'))
    .with_columns(panel=pl.format(
        '{} — {}  (n={})', pl.col('gene'),
        pl.col('phenotype').replace_strict(TRAIT_LABELS, default=pl.col('phenotype')),
        pl.col('n_variants'))))

# per-facet ordering: facets in PANELS order, bars ascending within a facet
order = (plot_df
         .with_columns(gene_rank=pl.col('gene').replace_strict({g: i for i, g in enumerate(PANELS)},
                                                               return_dtype=pl.Int32))
         .sort(['gene_rank', 'corr'])['plot_key'])
plot_df = plot_df.with_columns(
    pl.col('plot_key').cast(pl.Enum(order)),
    pl.col('panel').cast(pl.Enum(plot_df.unique(subset='gene', maintain_order=True)
                                 .sort(pl.col('gene').replace_strict({g: i for i, g in enumerate(PANELS)},
                                                                     return_dtype=pl.Int32))['panel'])))

# stars sit just above the bar (or above the axis for negative bars). `va='center_baseline'`
# instead of the default 'center': matplotlib centers a text glyph's *bounding box*, and '*' sits
# high inside its box (no descender), so 'center' visibly floats the star toward the top of its
# row -- 'center_baseline' aligns on the glyph's own baseline instead and sits on the bar.
pad = 0.04 * (plot_df['corr'].max() - min(plot_df['corr'].min(), 0))
star_df = plot_df.filter('star').with_columns(star_y=pl.max_horizontal('corr', pl.lit(0.0)) + pad)

# CATEGORY_COLORS = {'Computational': '#2A78D6', 'Experimental': '#afdc2e'}
CATEGORY_COLORS = {'Experimental': '#9F72BB', 'Computational': '#3DAED4'}

fig = (
    ggplot(plot_df, aes(x='plot_key', y='corr', fill='tool_type'))
    + geom_col(alpha=0.85, width=0.75)
    + geom_hline(yintercept=0, color='#444444', size=0.4)
    + geom_text(star_df, aes(x='plot_key', y='star_y'), label='*', size=14,
                va='center_baseline', inherit_aes=False)
    + facet_wrap('panel', nrow=2, scales='free')
    + scale_fill_manual(values=CATEGORY_COLORS, name=' ')
    + scale_x_discrete(labels=lambda ks: [k.split('||')[1] for k in ks])
    + labs(x='', y='Spearman correlation with trait\n(direction-corrected)')
    + coord_flip()
    + theme_minimal()
    + theme(figure_size=(2 * len(PANELS), 2.3 * len(PANELS)),
            legend_position='bottom',
            legend_direction='horizontal',
            legend_box_spacing=0.01,
            legend_margin=0,
            legend_text=element_text(size=12),
            legend_title=element_text(size=12),
            axis_text=element_text(size=10.5),
            axis_title=element_text(size=12, lineheight=1.4),
            strip_text=element_text(size=12, ha='right'),
            panel_spacing=0.03,
            plot_background=element_rect(fill='white', color='white'))
)

fig.save(f'{FIG_DIR}/F5_expAssays_4genes_correlations_allvall.svg', dpi=200, verbose=False)
fig